# Virtual 8086 Stepper Motor Laboratory

Interactive educational simulation for the 8086 stepper-motor interfacing experiment.

**Features:** clockwise / anticlockwise motion, variable step size, step count, adjustable delay,
rotor visualization, ALP listing viewer, and an agent-style execution log.

> Simulation only — no physical hardware is controlled.


In [ ]:
%pip -q install ipywidgets matplotlib


In [ ]:
import sys, time
from pathlib import Path
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# Import the reusable model when this notebook is run from the extracted project folder.
# In Colab, the fallback below keeps the notebook self-contained.
try:
    sys.path.insert(0, str(Path.cwd() / "src"))
    from stepper_simulator import StepperMotor, assembly_listing, SEQUENCE
except Exception:
    SEQUENCE = (0b1000, 0b0100, 0b0010, 0b0001)
    class StepperMotor:
        def __init__(self):
            self.state_index, self.position_steps = 0, 0
        def move(self, direction, steps):
            delta = 1 if direction == "clockwise" else -1
            events = []
            for _ in range(steps):
                self.state_index = (self.state_index + delta) % 4
                self.position_steps += delta
                events.append({"state": self.state_index,
                               "pattern": f"{SEQUENCE[self.state_index]:04b}",
                               "position_steps": self.position_steps})
            return events
    def assembly_listing(direction):
        rotate = "ROR AL, 1" if direction == "clockwise" else "ROL AL, 1"
        return f"""ORG 6000H
START: MOV AL, 80H
       MOV DX, 0FFC6H
       OUT DX, AL
       MOV AL, 88H
       MOV DX, 0FFC4H
       OUT DX, AL
       CALL DELAY
       {rotate}
       JMP START
DELAY: MOV CX, 2000H
WAIT:  LOOP WAIT
       RET"""

motor = StepperMotor()
print("Virtual laboratory ready.")


In [ ]:
direction = widgets.ToggleButtons(
    options=[("Clockwise", "clockwise"), ("Anticlockwise", "anticlockwise")],
    value="clockwise", description="Direction:")
units = widgets.IntSlider(value=12, min=1, max=100, step=1, description="Move units:")
step_size = widgets.IntSlider(value=1, min=1, max=8, step=1, description="Step size:")
delay_ms = widgets.IntSlider(value=0, min=0, max=250, step=10, description="Delay (ms):")
run_button = widgets.Button(description="Run simulation", button_style="success", icon="play")
reset_button = widgets.Button(description="Reset motor", button_style="warning", icon="refresh")
out = widgets.Output()

def draw_motor():
    # A compact top-view rotor diagram; the highlighted arm indicates current electrical state.
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.set_aspect("equal")
    ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, linewidth=2))
    ax.add_patch(plt.Circle((0, 0), 0.12, fill=True))
    import math
    angle = (motor.state_index * 90) % 360
    rad = math.radians(angle)
    ax.arrow(0, 0, 0.78*math.cos(rad), 0.78*math.sin(rad),
             head_width=0.12, length_includes_head=True)
    for i, label in enumerate(["A", "B", "C", "D"]):
        a = math.radians(i*90)
        ax.text(1.18*math.cos(a), 1.18*math.sin(a), label,
                ha="center", va="center", fontsize=12)
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.axis("off")
    ax.set_title(f"Rotor state {motor.state_index} | position {motor.position_steps} full-steps")
    plt.show()

def on_run(_):
    with out:
        clear_output(wait=True)
        chosen_direction = direction.value
        total_steps = units.value * step_size.value
        print("AGENT WORKFLOW")
        print(f"1. PLAN      Direction={chosen_direction}; units={units.value}; step-size={step_size.value}")
        print(f"2. VALIDATE  Requested transitions={total_steps}; settings are within allowed range.")
        print("3. EXECUTE   Applying the four-state coil sequence...")
        events = motor.move(chosen_direction, total_steps)
        for i, event in enumerate(events, 1):
            print(f"   Step {i:03d}: state={event['state']} coils={event['pattern']} position={event['position_steps']}")
            if delay_ms.value:
                time.sleep(delay_ms.value / 1000)
        print(f"4. VERIFY    Completed {len(events)} transitions. Final position={motor.position_steps}.")
        print("\n8086 ALP VIEWER")
        print(assembly_listing(chosen_direction))
        draw_motor()

def on_reset(_):
    global motor
    motor = StepperMotor()
    with out:
        clear_output(wait=True)
        print("Motor state reset to home position (0 full-steps).")
        draw_motor()

run_button.on_click(on_run)
reset_button.on_click(on_reset)
display(widgets.VBox([
    widgets.HTML("<h3>Motor control panel</h3>"),
    direction, units, step_size, delay_ms,
    widgets.HBox([run_button, reset_button]), out
]))


## How to interpret the controls

- **Move units** sets the requested movement count.
- **Step size** multiplies the movement count to determine the number of electrical transitions.
- **Delay** slows the simulation so individual transitions can be observed.
- The log shows the four-phase coil pattern, rotor state, and accumulated position.

## Hardware reminder

The ALP is a transcription of the supplied manual. The virtual step-size control is a software
feature; the original continuous-loop listing does not itself implement a keyboard-driven
variable-step interface. Refer to `docs/Hardware_Notes.md` before any physical experiment.
